In [5]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import string

# 1. Data Preparation (same as before)
try:
    with open("shakespeare.txt", "r", encoding="utf-8") as f:
        text = f.read()
except FileNotFoundError:
    print("Error: shakespeare.txt not found. Please download the dataset and place it in the same directory.")
    exit()

text = text.lower()
text = text.translate(str.maketrans('', '', string.punctuation))
vocab = sorted(list(set(text)))
char_to_index = {u:i for i, u in enumerate(vocab)}
index_to_char = np.array(vocab)

seq_length = 50
data = [char_to_index[c] for c in text]
n = len(data) - seq_length
train_data = data[:int(n * 0.8)]
val_data = data[int(n * 0.8):int(n * 0.9)]
test_data = data[int(n * 0.9):]

def create_sequences(data):
    xs = []
    ys = []
    for i in range(0, len(data) - seq_length, 1):
        x = data[i:i + seq_length]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

train_sequences, train_labels = create_sequences(train_data)
val_sequences, val_labels = create_sequences(val_data)
test_sequences, test_labels = create_sequences(test_data)


# 2. Model Building (modified to use SimpleRNN)
model = keras.Sequential([
    keras.layers.Embedding(len(vocab), 50, input_length=seq_length),
    keras.layers.SimpleRNN(128, return_sequences=True),
    keras.layers.SimpleRNN(128),
    keras.layers.Dense(len(vocab), activation='softmax')
])

# 3. Model Compilation (same as before)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 4. Training (Reduced epochs for faster runtime - adjust as needed)
model.fit(train_sequences, train_labels, epochs=2, batch_size=128, validation_data=(val_sequences, val_labels))


Epoch 1/2
32578/32578 [==============================] - 2364s 72ms/step - loss: 1.5518 - accuracy: 0.5300 - val_loss: 1.5320 - val_accuracy: 0.5353
Epoch 2/2
13851/32578 [===========>..................] - ETA: 23:09 - loss: 1.4109 - accuracy: 0.5675

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



32578/32578 [==============================] - 2718s 83ms/step - loss: 1.4011 - accuracy: 0.5699 - val_loss: 1.5123 - val_accuracy: 0.5414


In [20]:
def generate_text(model, start_index, length):
    # Initialize with a sequence of seq_length random characters.
    generated_text = [np.random.randint(0, len(vocab)) for _ in range(seq_length)]  
    generated_text.append(start_index) # Add the actual starting index

    for _ in range(10):
        input_seq = np.array([generated_text[-seq_length:]]) # correct slicing
        prediction = model.predict(input_seq)
        next_index = np.argmax(prediction[0])
        generated_text.append(next_index)
    return "".join([index_to_char[i] for i in generated_text]) # corrected index


start_index = np.random.randint(0, len(vocab))
generated = generate_text(model, start_index, 200)
print("Generated Text:\n", generated)

1/1 [==============================] - 0s 28ms/step
Generated Text:
 c2vnzmra1b3948uv 82ugvclndt9s1z97no99crw
7tdurjf1gu
    the c
